In [ ]:
!pip install instaloader

In [ ]:
# get target information (target_accounts)
import pandas as pd
target_accounts = pd.read_csv("target_accounts.csv")["username"].tolist()


In [ ]:
import instaloader

# Create an Instaloader instance
L = instaloader.Instaloader(
    dirname_pattern="stories/{target}",
    filename_pattern="{date_utc}_{shortcode}",
    download_videos=True,
    download_video_thumbnails=False,
    save_metadata=False
)

import os


# Log in — required to view stories (even your own)
#USERNAME = "itz_adrian_97"
#PASSWORD = "!"  # avoid hardcoding in real use, see note below



USERNAME = os.environ["IG_USERNAME"]
PASSWORD = os.environ["IG_PASSWORD"]



import time

try:
    L.load_session_from_file(USERNAME)
except FileNotFoundError:
    try:
        L.login(USERNAME, PASSWORD)
    except instaloader.exceptions.TwoFactorAuthRequiredException:
        for attempt in range(3):
            code = input("Enter 2FA code: ").strip()
            try:
                L.two_factor_login(code)
                break
            except instaloader.exceptions.BadCredentialsException:
                print("Invalid or expired code, try again.")
        else:
            raise SystemExit("Too many failed 2FA attempts.")
    L.save_session_to_file()

# Target account(s) whose stories you want to download
targets = ["woah.my"]  # replace with the account(s) you want

# Get numeric user IDs for the target profiles
profiles = [instaloader.Profile.from_username(L.context, name) for name in targets]
user_ids = [p.userid for p in profiles]

# Download stories for those profiles
for story in L.get_stories(userids=user_ids):
    for item in story.get_items():
        L.download_storyitem(item, target=story.owner_username)

print("Done downloading stories.")

# Reel extraction (latest or by Date Range)

In [ ]:

import instaloader
import time
import random
from datetime import datetime, timedelta, timezone
import os
# --- Setup ---
L = instaloader.Instaloader(
    dirname_pattern="reels/{target}",
    filename_pattern="{date_utc}_{shortcode}",
    download_videos=True,
    download_video_thumbnails=False,
    download_pictures=False,   # reels are video posts; skip static image dumps
    save_metadata=False,
    post_metadata_txt_pattern="{caption}",  # skip the .txt caption files, set a pattern if you want them
    max_connection_attempts=3,
)

USERNAME = os.environ["IG_USERNAME"]

# --- Load session (avoid re-triggering 2FA / login flags) ---
try:
    L.load_session_from_file(USERNAME)
except FileNotFoundError:
    PASSWORD = os.environ["IG_PASSWORD"] #PASSWORD = "your_password"
    try:
        L.login(USERNAME, PASSWORD)
    except instaloader.exceptions.TwoFactorAuthRequiredException:
        for attempt in range(3):
            code = input("Enter 2FA code: ").strip()
            try:
                L.two_factor_login(code)
                break
            except instaloader.exceptions.BadCredentialsException:
                print("Invalid or expired code, try again.")
        else:
            raise SystemExit("Too many failed 2FA attempts.")
    L.save_session_to_file()


def get_reels(target: str, since: datetime = None, only_latest: bool = True, lookahead: int = 6):
    """
    Fetch reels from a target profile, correctly handling pinned posts.
    - Pulls up to `lookahead` posts from the top (covers up to 3 possible pinned + buffer)
    - Sorts by actual date_utc rather than trusting feed order
    - since: only include reels posted on/after this UTC datetime (optional)
    - only_latest: if True, return just the single most recent reel by date
    """
    profile = instaloader.Profile.from_username(L.context, target)

    candidates = []
    for i, post in enumerate(profile.get_posts()):
        if i >= lookahead and not since:
            # Once we've checked enough posts to cover any pinned ones, stop —
            # unless we're doing a date-range pull, where we still need to keep paging
            break

        if not post.is_video:
            continue
        if getattr(post, "product_type", None) not in (None, "clips", "reels"):
            continue

        post_date = post.date_utc.replace(tzinfo=timezone.utc)

        if since and post_date < since and i >= lookahead:
            # Past the pinned-post window AND past the date cutoff — safe to stop
            break

        if since and post_date < since:
            continue  # still within pinned-post window, skip but keep scanning

        candidates.append(post)

    # Sort by true post date, newest first — this fixes pinned-post ordering issues
    candidates.sort(key=lambda p: p.date_utc, reverse=True)

    if only_latest:
        return candidates[:1]
    return candidates


def download_reels_for_account(target: str, since: datetime = None, only_latest: bool = True, delay_range=(15, 30)):
    try:
        reels = get_reels(target, since=since, only_latest=only_latest)
    except instaloader.exceptions.ProfileNotExistsException:
        print(f"[{target}] Profile does not exist, skipping.")
        return 0
    except instaloader.exceptions.ConnectionException as e:
        print(f"[{target}] Connection error, skipping: {e}")
        return 0

    if not reels:
        print(f"[{target}] No matching reels found.")
        return 0

    downloaded = 0
    for post in reels:
        print(f"[{target}] Downloading reel {post.shortcode} posted {post.date_utc} UTC")
        try:
            L.download_post(post, target=target)
            downloaded += 1
        except Exception as e:
            print(f"[{target}] Failed to download {post.shortcode}: {e}")
        time.sleep(random.uniform(*delay_range))  # avoid rate limiting between downloads

    print(f"[{target}] Done. Downloaded {downloaded} reel(s).")
    return downloaded


def download_reels(target_accounts: list[str], since: datetime = None, only_latest: bool = True,
                    delay_range=(15, 30), account_delay_range=(30, 60)):
    """
    Download reels for a list of target accounts.
    - target_accounts: list of usernames to scrape
    - account_delay_range: extra pause between switching accounts, on top of per-download delay
    """
    total_downloaded = 0
    for i, target in enumerate(target_accounts):
        count = download_reels_for_account(target, since=since, only_latest=only_latest, delay_range=delay_range)
        total_downloaded += count

        # Pause between accounts (skip after the last one)
        if i < len(target_accounts) - 1:
            pause = random.uniform(*account_delay_range)
            print(f"Pausing {pause:.0f}s before next account...")
            time.sleep(pause)

    print(f"\nAll done. Downloaded {total_downloaded} reel(s) across {len(target_accounts)} account(s).")


if __name__ == "__main__":
    if target_accounts is None:
        target_accounts = ["chellemakesfood"] # demo account
    
        #target_accounts = ["woah.my","chellemakesfood"] # demo account

    # Example 1: latest reel from each account
    download_reels(target_accounts, only_latest=True)

    # Example 2: all reels from last 7 days for each account
    # cutoff = datetime.now(timezone.utc) - timedelta(days=7)
    # download_reels(target_accounts, since=cutoff, only_latest=False)

Loaded session from C:\Users\This\AppData\Local\Instaloader\session-itz_adrian_97.
[chellemakesfood] Downloading reel DcQgs8nBC6W posted 2026-08-20 10:08:58 UTC
[nostalgic food date? my tips …] reels/chellemakesfood\2026-08-20_10-08-58_DcQgs8nBC6W.mp4 exists 
[chellemakesfood] Done. Downloaded 1 reel(s).

All done. Downloaded 1 reel(s) across 1 account(s).


# Audio Transcriber

In [ ]:
!pip install faster-whisper

In [1]:
import json
from pathlib import Path
from faster_whisper import WhisperModel

MODEL_SIZE = "small"
model = WhisperModel(MODEL_SIZE, device="cpu", compute_type="int8")

INDEX_PATH = Path("reels_transcripts_index.json")


def transcribe(video_path: Path) -> dict:
    """Transcribe a video's audio directly — faster-whisper decodes via PyAV internally,
    no ffmpeg binary or separate audio extraction step needed."""
    segments, info = model.transcribe(str(video_path), beam_size=5)

    segment_list = []
    full_text_parts = []
    for seg in segments:
        segment_list.append({
            "start": round(seg.start, 2),
            "end": round(seg.end, 2),
            "text": seg.text.strip()
        })
        full_text_parts.append(seg.text.strip())

    return {
        "video_path": str(video_path),
        "language": info.language,
        "language_probability": round(info.language_probability, 3),
        "full_text": " ".join(full_text_parts),
        "segments": segment_list,
    }


def load_index() -> list[dict]:
    if INDEX_PATH.exists():
        return json.loads(INDEX_PATH.read_text())
    return []


def save_index(index: list[dict]):
    INDEX_PATH.write_text(json.dumps(index, indent=2, ensure_ascii=False))


def transcribe_new_videos(reels_dir: str = "reels"):
    index = load_index()
    indexed_paths = {entry["video_path"] for entry in index}

    video_files = list(Path(reels_dir).rglob("*.mp4"))
    new_count = 0

    for video_path in video_files:
        if str(video_path) in indexed_paths:
            continue

        print(f"Transcribing {video_path}...")
        try:
            result = transcribe(video_path)
            index.append(result)
            new_count += 1
        except Exception as e:
            print(f"Failed to transcribe {video_path}: {e}")

    save_index(index)
    print(f"Transcribed {new_count} new video(s). Index now has {len(index)} entries.")
    return index


def search(keywords: list[str], match_all: bool = False, case_sensitive: bool = False) -> list[dict]:
    index = load_index()
    results = []

    for entry in index:
        text = entry["full_text"] if case_sensitive else entry["full_text"].lower()
        search_terms = keywords if case_sensitive else [k.lower() for k in keywords]

        matched_terms = [term for term in search_terms if term in text]
        is_match = (len(matched_terms) == len(search_terms)) if match_all else (len(matched_terms) > 0)
        if not is_match:
            continue

        matching_segments = []
        for seg in entry["segments"]:
            seg_text = seg["text"] if case_sensitive else seg["text"].lower()
            if any(term in seg_text for term in matched_terms):
                matching_segments.append(seg)

        results.append({
            "video_path": entry["video_path"],
            "matched_keywords": matched_terms,
            "matching_segments": matching_segments,
        })

    return results


def print_search_results(keywords: list[str], match_all: bool = False):
    results = search(keywords, match_all=match_all)
    if not results:
        print(f"No matches found for {keywords}.")
        return

    print(f"Found {len(results)} matching video(s) for {keywords}:\n")
    for r in results:
        print(f"📹 {r['video_path']}")
        print(f"   Matched: {r['matched_keywords']}")
        for seg in r["matching_segments"]:
            print(f"   [{seg['start']}s - {seg['end']}s] {seg['text']}")
        print()


if __name__ == "__main__":
    transcribe_new_videos(reels_dir="reels")
    print_search_results(["family", "kid"], match_all=False)

: 